<a href="https://colab.research.google.com/github/jigyasaG01/cattle_breed/blob/main/bcs.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
!unzip "/content/bcs_dz.zip" -d BCS_data


Archive:  /content/bcs_dz.zip
   creating: BCS_data/bcs_d/
   creating: BCS_data/bcs_d/fat/
  inflating: BCS_data/bcs_d/fat/Bhadawari_002.jpg  
  inflating: BCS_data/bcs_d/fat/Bhadawari_009.jpg  
  inflating: BCS_data/bcs_d/fat/Bhadawari_011.jpg  
  inflating: BCS_data/bcs_d/fat/Bhadawari_012.jpg  
  inflating: BCS_data/bcs_d/fat/Bhadawari_016.jpg  
  inflating: BCS_data/bcs_d/fat/Gir_003.jpg  
  inflating: BCS_data/bcs_d/fat/Gir_012.jpg  
  inflating: BCS_data/bcs_d/fat/Gir_021.jpg  
  inflating: BCS_data/bcs_d/fat/Gir_041.jpg  
  inflating: BCS_data/bcs_d/fat/Gir_042.jpg  
  inflating: BCS_data/bcs_d/fat/Hariana_014.jpg  
  inflating: BCS_data/bcs_d/fat/Hariana_018.jpg  
  inflating: BCS_data/bcs_d/fat/Hariana_022.jpg  
  inflating: BCS_data/bcs_d/fat/Hariana_024.jpg  
  inflating: BCS_data/bcs_d/fat/Kankrej_002.jpg  
  inflating: BCS_data/bcs_d/fat/Kankrej_013.jpg  
  inflating: BCS_data/bcs_d/fat/Kankrej_024.jpg  
  inflating: BCS_data/bcs_d/fat/Kankrej_034.jpg  
  inflating: BCS_d

In [ ]:
!ls BCS_data/bcs_d



fat  moderate  thin


In [ ]:
!for d in BCS_data/bcs_d/*; do echo "$(basename $d): $(ls $d | wc -l)"; done




fat: 55
moderate: 129
thin: 54


In [ ]:
import os, shutil, random

SRC = "BCS_data/bcs_d"
DEST = "BCS_final"
VAL_RATIO = 0.2

for cls in os.listdir(SRC):
    cls_path = os.path.join(SRC, cls)
    if not os.path.isdir(cls_path):
        continue

    os.makedirs(f"{DEST}/train/{cls}", exist_ok=True)
    os.makedirs(f"{DEST}/val/{cls}", exist_ok=True)

    images = os.listdir(cls_path)
    random.shuffle(images)

    val_count = int(len(images) * VAL_RATIO)

    # train images
    for img in images[val_count:]:
        shutil.copy(
            os.path.join(cls_path, img),
            f"{DEST}/train/{cls}/{img}"
        )

    # val images
    for img in images[:val_count]:
        shutil.copy(
            os.path.join(cls_path, img),
            f"{DEST}/val/{cls}/{img}"
        )


In [ ]:
!ls BCS_final/train
!ls BCS_final/val


fat  moderate  thin
fat  moderate  thin


In [ ]:
!for d in BCS_final/train/*; do echo "TRAIN $(basename $d): $(ls $d | wc -l)"; done
!for d in BCS_final/val/*; do echo "VAL $(basename $d): $(ls $d | wc -l)"; done


TRAIN fat: 44
TRAIN moderate: 104
TRAIN thin: 44
VAL fat: 11
VAL moderate: 25
VAL thin: 10


In [ ]:
from torchvision import datasets, transforms

train_tfms = transforms.Compose([
    transforms.Resize((224,224)),
    transforms.ToTensor(),
])

val_tfms = transforms.Compose([
    transforms.Resize((224,224)),
    transforms.ToTensor(),
])

train_ds = datasets.ImageFolder("BCS_final/train", transform=train_tfms)
val_ds   = datasets.ImageFolder("BCS_final/val", transform=val_tfms)

print("Classes:", train_ds.classes)
print("Train samples:", len(train_ds))
print("Val samples:", len(val_ds))


Classes: ['fat', 'moderate', 'thin']
Train samples: 192
Val samples: 46


In [ ]:
import torch
from torch.utils.data import DataLoader

train_loader = DataLoader(train_ds, batch_size=16, shuffle=True)
val_loader   = DataLoader(val_ds, batch_size=16, shuffle=False)

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
device


device(type='cpu')

In [ ]:
from torchvision import models
from torch import nn, optim

model = models.resnet18(pretrained=True)

# freeze backbone
for p in model.parameters():
    p.requires_grad = False

# replace classifier
model.fc = nn.Linear(model.fc.in_features, 3)

model = model.to(device)


/usr/local/lib/python3.12/dist-packages/torchvision/models/_utils.py:208: UserWarning: The parameter 'pretrained' is deprecated since 0.13 and may be removed in the future, please use 'weights' instead.
  warnings.warn(
/usr/local/lib/python3.12/dist-packages/torchvision/models/_utils.py:223: UserWarning: Arguments other than a weight enum or `None` for 'weights' are deprecated since 0.13 and may be removed in the future. The current behavior is equivalent to passing `weights=ResNet18_Weights.IMAGENET1K_V1`. You can also use `weights=ResNet18_Weights.DEFAULT` to get the most up-to-date weights.
  warnings.warn(msg)


Downloading: "https://download.pytorch.org/models/resnet18-f37072fd.pth" to /root/.cache/torch/hub/checkpoints/resnet18-f37072fd.pth


100%|██████████| 44.7M/44.7M [00:00<00:00, 51.3MB/s]


In [ ]:
criterion = nn.CrossEntropyLoss()
optimizer = optim.Adam(model.fc.parameters(), lr=0.001)


In [ ]:
epochs = 10

for epoch in range(epochs):
    model.train()
    running_loss = 0

    for imgs, labels in train_loader:
        imgs, labels = imgs.to(device), labels.to(device)

        optimizer.zero_grad()
        outputs = model(imgs)
        loss = criterion(outputs, labels)
        loss.backward()
        optimizer.step()

        running_loss += loss.item()

    # validation
    model.eval()
    correct, total = 0, 0
    with torch.no_grad():
        for imgs, labels in val_loader:
            imgs, labels = imgs.to(device), labels.to(device)
            outputs = model(imgs)
            _, preds = torch.max(outputs, 1)
            correct += (preds == labels).sum().item()
            total += labels.size(0)

    acc = 100 * correct / total
    print(f"Epoch {epoch+1}/{epochs} | Loss: {running_loss:.2f} | Val Acc: {acc:.2f}%")


Epoch 1/10 | Loss: 13.85 | Val Acc: 47.83%
Epoch 2/10 | Loss: 11.69 | Val Acc: 52.17%
Epoch 3/10 | Loss: 11.02 | Val Acc: 52.17%
Epoch 4/10 | Loss: 10.45 | Val Acc: 47.83%
Epoch 5/10 | Loss: 9.84 | Val Acc: 52.17%
Epoch 6/10 | Loss: 9.31 | Val Acc: 52.17%
Epoch 7/10 | Loss: 9.33 | Val Acc: 36.96%
Epoch 8/10 | Loss: 8.71 | Val Acc: 52.17%
Epoch 9/10 | Loss: 8.32 | Val Acc: 39.13%
Epoch 10/10 | Loss: 8.34 | Val Acc: 54.35%


In [ ]:
torch.save(model.state_dict(), "bcs_model.pth")


In [ ]:
from PIL import Image
import torch.nn.functional as F

def predict_bcs(img_path):
    img = Image.open(img_path).convert("RGB")
    img = val_tfms(img).unsqueeze(0).to(device)

    model.eval()
    with torch.no_grad():
        out = model(img)
        probs = F.softmax(out, dim=1)
        conf, pred = torch.max(probs, 1)

    classes = train_ds.classes
    return classes[pred.item()], conf.item()


In [ ]:
bcs, confidence = predict_bcs("/content/cattle.jpg")
print(f"Approx BCS: {bcs} ({confidence*100:.2f}%)")


Approx BCS: moderate (83.66%)
